# summerize your data

In [29]:
import requests
import getpass
import json
import pandas as pd
from typing import Dict, List, Tuple
from collections import Counter
import os

### Access NOMAD

In [30]:
# --- Function to get an access token from the NOMAD API ---
def get_access_token(
    username: str, password: str, url: str, timeout_in_sec: int = 10
) -> str:
    """Authenticates with the NOMAD API and returns an access token."""
    response = requests.post(
        f"{url}/auth/token",
        data={"username": username, "password": password},
        timeout=timeout_in_sec,
    )
    response.raise_for_status()  # This will raise an error if login fails
    return response.json()["access_token"]

# --- Configuration ---
URL = "https://nomad.nanolab.dtu.dk/nomad-oasis/api/v1"

# --- Connect and Prepare Headers for Future Use ---
access_token = getpass.getpass("Enter your NOMAD personal access token: ").strip()
if not access_token:
    raise ValueError("A NOMAD personal access token is required.")
headers = {"Authorization": f"Bearer {access_token}"}
print("Authorization header configured; the API connection will be checked on the first request.")

ValueError: A NOMAD personal access token is required.

# Summary table of your samples and their characterization
### just run this cell
defining the search function

In [32]:
# Get sputtering characterization table - FINAL VERSION with position extraction
def get_sputtering_characterization_table_v5(url: str, headers: dict, author_name: str = "Lena Mittmann") -> pd.DataFrame:
    """
    Query NOMAD API to find all sputtering depositions by a specific author
    and identify which characterization methods (EDX, XRD, XPS) are available.
    Extracts substrate position from CHARACTERIZATION entry names.
    
    Returns a DataFrame with columns: Sample Name, Position, Entry ID, Upload ID, EDX, XRD, XPS
    """
    
    print(f"Fetching sputtering depositions for author: {author_name}")
    print("="*80)
    
    # Query to get all DTUSputtering entries
    all_sputtering = []
    page_after_value = None
    
    while True:
        query = {
            "owner": "visible",
            "query": {
                "and": [
                    {"entry_type:all": ["DTUSputtering"]},
                    {"authors.name": author_name}
                ]
            },
            "pagination": {
                "page_size": 1000,
                "order_by": "entry_id",
                "page_after_value": page_after_value
            },
            "required": {
                "include": ["entry_id", "entry_name", "upload_id", "authors"]
            }
        }
        
        try:
            response = requests.post(
                f"{url}/entries/query",
                headers=headers,
                json=query,
                timeout=90,
            )
            response.raise_for_status()
            response_json = response.json()
            
            page_data = response_json.get("data", [])
            if not page_data:
                break
            
            all_sputtering.extend(page_data)
            
            print(f"Processed {len(page_data)} entries, found {len(all_sputtering)} total so far...")
            
            pagination = response_json.get("pagination", {})
            page_after_value = pagination.get("next_page_after_value")
            
            if not page_after_value:
                break
            
        except requests.exceptions.RequestException as e:
            raise RuntimeError(f"NOMAD query for sputtering depositions failed: {e}") from e
    
    print(f"\n✅ Found {len(all_sputtering)} sputtering depositions by {author_name}")
    print("="*80)
    
    if not all_sputtering:
        print("\n⚠️ No sputtering depositions found. Returning empty table.")
        return pd.DataFrame(columns=["Sample Name", "Position", "Entry ID", "Upload ID", "EDX", "XRD", "XPS"])
    
    # Fetch ALL entries by this author INCLUDING characterization entry names
    print("\n📊 Fetching all characterization entries with details...")
    
    all_author_entries = []
    page_after_value = None
    
    while True:
        query = {
            "owner": "visible",
            "query": {
                "authors.name": author_name
            },
            "pagination": {
                "page_size": 1000,
                "order_by": "entry_id",
                "page_after_value": page_after_value
            },
            "required": {
                "include": ["entry_id", "entry_type", "entry_name", "upload_id"]
            }
        }
        
        try:
            response = requests.post(
                f"{url}/entries/query",
                headers=headers,
                json=query,
                timeout=90,
            )
            response.raise_for_status()
            response_json = response.json()
            
            page_data = response_json.get("data", [])
            if not page_data:
                break
            
            all_author_entries.extend(page_data)
            
            pagination = response_json.get("pagination", {})
            page_after_value = pagination.get("next_page_after_value")
            
            if not page_after_value:
                break
            
        except requests.exceptions.RequestException as e:
            raise RuntimeError(f"NOMAD query for characterization entries failed: {e}") from e
    
    print(f"✅ Found {len(all_author_entries)} total entries by {author_name}")
    
    # Group characterization entries by upload_id, entry_type, AND position
    # Structure: upload_characterizations[upload_id][position] = {"EDX": bool, "XRD": bool, "XPS": bool}
    upload_characterizations = {}
    
    position_codes = ['FL', 'FR', 'BL', 'BR', 'C', 'TL', 'TR', 'G', 'T', 'B', 'L', 'R', 'b', 'c', 'g', 't']
    
    for entry in all_author_entries:
        upload_id = entry.get("upload_id")
        entry_type = entry.get("entry_type")
        entry_name = entry.get("entry_name", "")
        
        # Only process characterization entries
        if entry_type not in ["EDXMeasurement", "DTUXRDMeasurement", "XPSMeasurement"]:
            continue
        
        # Extract position from entry name (4th element when split by space)
        # Example: "anait 0016 Ba FL XRD" -> position is "FL"
        parts = entry_name.split(" ")
        position = "Unknown"
        
        # Look for position code in the parts
        for part in parts:
            if part in position_codes:
                position = part
                break
        
        # Initialize nested structure if needed
        if upload_id not in upload_characterizations:
            upload_characterizations[upload_id] = {}
        
        if position not in upload_characterizations[upload_id]:
            upload_characterizations[upload_id][position] = {
                "EDX": False,
                "XRD": False,
                "XPS": False
            }
        
        # Mark the characterization type
        if entry_type == "EDXMeasurement":
            upload_characterizations[upload_id][position]["EDX"] = True
        elif entry_type == "DTUXRDMeasurement":
            upload_characterizations[upload_id][position]["XRD"] = True
        elif entry_type == "XPSMeasurement":
            upload_characterizations[upload_id][position]["XPS"] = True
    
    print(f"✅ Found characterizations across {len(upload_characterizations)} uploads")
    
    # Now create rows: one per sputtering entry per position
    print("\n📊 Building table with position-specific characterizations...")
    results = []
    
    for i, sputtering_entry in enumerate(all_sputtering, 1):
        entry_id = sputtering_entry.get("entry_id", "Unknown")
        entry_name = sputtering_entry.get("entry_name", "Unknown")
        upload_id = sputtering_entry.get("upload_id")
        
        # Check if this upload has characterizations
        if upload_id in upload_characterizations:
            # Create a row for each position found in characterizations
            positions_dict = upload_characterizations[upload_id]
            
            for position, char_types in positions_dict.items():
                results.append({
                    "Sample Name": entry_name,
                    "Position": position,
                    "Entry ID": entry_id,
                    "Upload ID": upload_id,
                    "EDX": "Yes" if char_types["EDX"] else "No",
                    "XRD": "Yes" if char_types["XRD"] else "No",
                    "XPS": "Yes" if char_types["XPS"] else "No"
                })
        else:
            # No characterizations found for this upload
            results.append({
                "Sample Name": entry_name,
                "Position": "No characterization",
                "Entry ID": entry_id,
                "Upload ID": upload_id,
                "EDX": "No",
                "XRD": "No",
                "XPS": "No"
            })
        
        if i % 10 == 0:
            print(f"  Processed {i}/{len(all_sputtering)} sputtering entries...")
    
    # Create DataFrame
    df = pd.DataFrame(results)
    
    # Summary statistics
    print("\n" + "="*80)
    print("SUMMARY")
    print("="*80)
    print(f"Total sputtering depositions: {len(all_sputtering)}")
    print(f"Total rows (deposition × position): {len(df)}")
    if len(df) > 0:
        print(f"With EDX: {(df['EDX'] == 'Yes').sum()}")
        print(f"With XRD: {(df['XRD'] == 'Yes').sum()}")
        print(f"With XPS: {(df['XPS'] == 'Yes').sum()}")
        print(f"With any characterization: {((df['EDX'] == 'Yes') | (df['XRD'] == 'Yes') | (df['XPS'] == 'Yes')).sum()}")
        
        # Show position breakdown
        print(f"\nPositions found:")
        position_counts = df['Position'].value_counts()
        for position, count in position_counts.items():
            print(f"  • {position}: {count} rows")
    print("="*80)
    
    return df



### Change AUTHOR_NAME to your name
the CSV will be saved at github > code in progress > Anat > API

In [33]:

# ============================================================================
# ⚠️⚠️⚠️ CHANGE AUTHOR NAME HERE ⚠️⚠️⚠️
# Available options: "Anat Itzhak", "Lena Mittmann", "Inês Diogo"
# ============================================================================
AUTHOR_NAME = "Lena Mittmann"  # ← ← ← CHANGE THIS LINE TO CHANGE AUTHOR
# ============================================================================

print("="*80)
print(f"🔍 QUERYING DATA FOR AUTHOR: {AUTHOR_NAME}")
print("="*80)
print()

try:
    characterization_df = get_sputtering_characterization_table_v5(URL, headers, AUTHOR_NAME)
    
    # Display the table
    if len(characterization_df) > 0:
        print("\n📊 CHARACTERIZATION TABLE (with positions):")
        print(characterization_df.to_string(index=False))
        
        # Save to CSV - filename based on author name
        safe_filename = AUTHOR_NAME.lower().replace(" ", "_").replace("ê", "e").replace("í", "i")
        output_file = f"{safe_filename}_sputtering_characterization_with_positions.csv"
        characterization_df.to_csv(output_file, index=False)
        
        print("\n" + "="*80)
        print(f"✅ Table saved to: {output_file}")
        print(f"✅ Data retrieved for author: {AUTHOR_NAME}")
        print(f"✅ Table saved to: {output_file}")
        print(f"📁 Full path: {os.path.abspath(output_file)}")
        print("="*80)
    else:
        print("\n⚠️ No data to display or save.")
    
except NameError:
    print("❌ Error: Make sure you have run the first cell to define 'URL' and 'headers'.")
except Exception as e:
    print(f"❌ An unexpected error occurred: {e}")
    import traceback
    traceback.print_exc()

🔍 QUERYING DATA FOR AUTHOR: Lena Mittmann

Fetching sputtering depositions for author: Lena Mittmann
Processed 64 entries, found 64 total so far...

✅ Found 64 sputtering depositions by Lena Mittmann

📊 Fetching all characterization entries with details...
✅ Found 10422 total entries by Lena Mittmann
✅ Found characterizations across 89 uploads

📊 Building table with position-specific characterizations...
  Processed 10/64 sputtering entries...
  Processed 20/64 sputtering entries...
  Processed 30/64 sputtering entries...
  Processed 40/64 sputtering entries...
  Processed 50/64 sputtering entries...
  Processed 60/64 sputtering entries...

SUMMARY
Total sputtering depositions: 64
Total rows (deposition × position): 238
With EDX: 234
With XRD: 59
With XPS: 0
With any characterization: 234

Positions found:
  • BL: 59 rows
  • FR: 59 rows
  • BR: 58 rows
  • FL: 57 rows
  • No characterization: 4 rows
  • B: 1 rows

📊 CHARACTERIZATION TABLE (with positions):
    Sample Name            P

# Find samples missing a specific characterization type
### just run this cell
Defining the function

In [25]:
# Find sputtering samples without EDX and XRD - COMPLETE ANALYSIS
print("🔍 Complete characterization analysis for sputtering samples")
print("="*80)

if 'characterization_df' in dir() and len(characterization_df) > 0:
    # Filter for samples that don't have BOTH EDX and XRD
    missing_both = characterization_df[
        (characterization_df['EDX'] == 'No') & 
        (characterization_df['XRD'] == 'No')
    ]
    
    # Filter for samples missing EDX only (have XRD but no EDX)
    missing_edx = characterization_df[
        (characterization_df['EDX'] == 'No') & 
        (characterization_df['XRD'] == 'Yes')
    ]
    
    # Filter for samples missing XRD only (have EDX but no XRD)
    missing_xrd = characterization_df[
        (characterization_df['EDX'] == 'Yes') & 
        (characterization_df['XRD'] == 'No')
    ]
    
    # Filter for samples with ONLY EDX (no XRD, no XPS)
    only_edx = characterization_df[
        (characterization_df['EDX'] == 'Yes') & 
        (characterization_df['XRD'] == 'No') & 
        (characterization_df['XPS'] == 'No')
    ]
    
    # Filter for samples with ONLY XRD (no EDX, no XPS)
    only_xrd = characterization_df[
        (characterization_df['EDX'] == 'No') & 
        (characterization_df['XRD'] == 'Yes') & 
        (characterization_df['XPS'] == 'No')
    ]
    
    # Filter for samples with BOTH EDX and XRD (regardless of XPS)
    both_edx_xrd = characterization_df[
        (characterization_df['EDX'] == 'Yes') & 
        (characterization_df['XRD'] == 'Yes')
    ]
    
    # Filter for samples with NO characterization at all
    no_characterization = characterization_df[
        (characterization_df['EDX'] == 'No') & 
        (characterization_df['XRD'] == 'No') & 
        (characterization_df['XPS'] == 'No')
    ]
    
    print("\n📊 SAMPLES WITH BOTH EDX AND XRD:")
    print("="*80)
    if len(both_edx_xrd) > 0:
        print(f"✅ Found {len(both_edx_xrd)} samples with both EDX and XRD:\n")
        for idx, row in both_edx_xrd.iterrows():
            xps_status = "with XPS" if row['XPS'] == 'Yes' else "no XPS"
            print(f"• {row['Sample Name']} - Position: {row['Position']} ({xps_status})")
    else:
        print("❌ No samples have both EDX and XRD")
    
    print("\n📊 SAMPLES WITH ONLY EDX (no XRD, no XPS):")
    print("="*80)
    if len(only_edx) > 0:
        print(f"Found {len(only_edx)} samples with only EDX:\n")
        for idx, row in only_edx.iterrows():
            print(f"• {row['Sample Name']} - Position: {row['Position']}")
    else:
        print("✅ No samples have only EDX")
    
    print("\n📊 SAMPLES WITH ONLY XRD (no EDX, no XPS):")
    print("="*80)
    if len(only_xrd) > 0:
        print(f"Found {len(only_xrd)} samples with only XRD:\n")
        for idx, row in only_xrd.iterrows():
            print(f"• {row['Sample Name']} - Position: {row['Position']}")
    else:
        print("✅ No samples have only XRD")
    
    print("\n📊 SAMPLES MISSING BOTH EDX AND XRD:")
    print("="*80)
    if len(missing_both) > 0:
        print(f"Found {len(missing_both)} samples without both EDX and XRD:\n")
        for idx, row in missing_both.iterrows():
            xps_status = f"(has XPS)" if row['XPS'] == 'Yes' else "(no characterization)"
            print(f"• {row['Sample Name']} - Position: {row['Position']} {xps_status}")
    else:
        print("✅ All samples have either EDX or XRD (or both)!")
    
    print("\n📊 SAMPLES MISSING EDX (but have XRD):")
    print("="*80)
    if len(missing_edx) > 0:
        print(f"Found {len(missing_edx)} samples without EDX:\n")
        for idx, row in missing_edx.iterrows():
            xps_status = "with XPS" if row['XPS'] == 'Yes' else "no XPS"
            print(f"• {row['Sample Name']} - Position: {row['Position']} ({xps_status})")
    else:
        print("✅ No samples missing only EDX")
    
    print("\n📊 SAMPLES MISSING XRD (but have EDX):")
    print("="*80)
    if len(missing_xrd) > 0:
        print(f"Found {len(missing_xrd)} samples without XRD:\n")
        for idx, row in missing_xrd.iterrows():
            xps_status = "with XPS" if row['XPS'] == 'Yes' else "no XPS"
            print(f"• {row['Sample Name']} - Position: {row['Position']} ({xps_status})")
    else:
        print("✅ No samples missing only XRD")
    
    print("\n📊 SAMPLES WITH NO CHARACTERIZATION AT ALL:")
    print("="*80)
    if len(no_characterization) > 0:
        print(f"Found {len(no_characterization)} samples with no characterization:\n")
        for idx, row in no_characterization.iterrows():
            print(f"• {row['Sample Name']} - Position: {row['Position']}")
    else:
        print("✅ All samples have at least one characterization!")
    
    # Summary statistics
    print("\n" + "="*80)
    print("SUMMARY STATISTICS:")
    print("="*80)
    total = len(characterization_df)
    
    print(f"Total samples (with positions): {total}")
    print(f"\nCharacterization coverage:")
    print(f"  ✅ BOTH EDX and XRD: {len(both_edx_xrd)} ({100*len(both_edx_xrd)/total:.1f}%)")
    print(f"  📊 ONLY EDX (no XRD, no XPS): {len(only_edx)} ({100*len(only_edx)/total:.1f}%)")
    print(f"  📊 ONLY XRD (no EDX, no XPS): {len(only_xrd)} ({100*len(only_xrd)/total:.1f}%)")
    print(f"  ⚠️  Missing BOTH EDX and XRD: {len(missing_both)} ({100*len(missing_both)/total:.1f}%)")
    print(f"  ⚠️  Missing only EDX: {len(missing_edx)} ({100*len(missing_edx)/total:.1f}%)")
    print(f"  ⚠️  Missing only XRD: {len(missing_xrd)} ({100*len(missing_xrd)/total:.1f}%)")
    print(f"  ❌ NO characterization: {len(no_characterization)} ({100*len(no_characterization)/total:.1f}%)")
    print("="*80)
    
    # Save to CSV files
    if len(missing_both) > 0:
        missing_both.to_csv("samples_missing_edx_and_xrd.csv", index=False)
        print(f"\n💾 Samples missing both EDX and XRD saved to: samples_missing_edx_and_xrd.csv")
    
    if len(only_edx) > 0:
        only_edx.to_csv("samples_only_edx.csv", index=False)
        print(f"💾 Samples with only EDX saved to: samples_only_edx.csv")
    
    if len(only_xrd) > 0:
        only_xrd.to_csv("samples_only_xrd.csv", index=False)
        print(f"💾 Samples with only XRD saved to: samples_only_xrd.csv")
    
    if len(no_characterization) > 0:
        no_characterization.to_csv("samples_no_characterization.csv", index=False)
        print(f"💾 Samples with no characterization saved to: samples_no_characterization.csv")
    
else:
    print("❌ characterization_df not found. Please run the sputtering characterization table cell first.")

🔍 Complete characterization analysis for sputtering samples

📊 SAMPLES WITH BOTH EDX AND XRD:
✅ Found 59 samples with both EDX and XRD:

• mittma 0003 Cu - Position: BR (no XPS)
• mittma 0020 Cu - Position: BR (no XPS)
• mittma 0020 Cu - Position: FL (no XPS)
• mittma 0020 Cu - Position: FR (no XPS)
• mittma 0018 Cu - Position: B (no XPS)
• mittma 0017 Cu - Position: FL (no XPS)
• mittma 0017 Cu - Position: BL (no XPS)
• mittma 0017 Cu - Position: FR (no XPS)
• mittma 0017 Cu - Position: BR (no XPS)
• mittma 0012 Cu - Position: FR (no XPS)
• mittma 0012 Cu - Position: BR (no XPS)
• mittma 0012 Cu - Position: BL (no XPS)
• mittma 0012 Cu - Position: FL (no XPS)
• mittma 0029 Cu - Position: BR (no XPS)
• mittma 0029 Cu - Position: FL (no XPS)
• mittma 0029 Cu - Position: FR (no XPS)
• mittma 0029 Cu - Position: BL (no XPS)
• mittma 0021 Cu - Position: BR (no XPS)
• mittma 0013 Cu - Position: BR (no XPS)
• mittma 0013 Cu - Position: FR (no XPS)
• mittma 0013 Cu - Position: FL (no XPS)
• m

### Set AUTHOR_NAME and CHARACTERIZATION_TYPE to search for samples missing that characterization

In [27]:
# ============================================================================
# ⚠️⚠️⚠️ SET YOUR PARAMETERS HERE ⚠️⚠️⚠️
# ============================================================================
AUTHOR_NAME = "Lena Mittmann"  # ← Change to your author name
CHARACTERIZATION_TYPE = "XRD"  # ← Change to: "EDX", "XRD", or "XPS"
# ============================================================================

print("="*80)
print(f"🔍 Searching for samples by {AUTHOR_NAME}")
print(f"📊 Missing characterization type: {CHARACTERIZATION_TYPE}")
print("="*80)

if 'characterization_df' in dir() and len(characterization_df) > 0:
    # Filter for samples missing the specified characterization type
    missing_char = characterization_df[characterization_df[CHARACTERIZATION_TYPE] == 'No'].copy()
    
    if len(missing_char) > 0:
        print(f"\n📋 Found {len(missing_char)} samples WITHOUT {CHARACTERIZATION_TYPE}:\n")
        print("="*80)
        
        # Display each sample with what characterizations it DOES have
        for idx, row in missing_char.iterrows():
            available_chars = []
            if row['EDX'] == 'Yes':
                available_chars.append('EDX')
            if row['XRD'] == 'Yes':
                available_chars.append('XRD')
            if row['XPS'] == 'Yes':
                available_chars.append('XPS')
            
            if available_chars:
                other_chars = f"Has: {', '.join(available_chars)}"
            else:
                other_chars = "No characterization at all"
            
            print(f"• {row['Sample Name']} - Position: {row['Position']}")
            print(f"  └─ {other_chars}")
        
        # Summary statistics
        print("\n" + "="*80)
        print("SUMMARY:")
        print("="*80)
        total = len(characterization_df)
        missing_count = len(missing_char)
        
        print(f"Total samples: {total}")
        print(f"Missing {CHARACTERIZATION_TYPE}: {missing_count} ({100*missing_count/total:.1f}%)")
        print(f"Have {CHARACTERIZATION_TYPE}: {total - missing_count} ({100*(total-missing_count)/total:.1f}%)")
        
        # Count how many have other characterizations
        has_edx = (missing_char['EDX'] == 'Yes').sum()
        has_xrd = (missing_char['XRD'] == 'Yes').sum()
        has_xps = (missing_char['XPS'] == 'Yes').sum()
        has_nothing = ((missing_char['EDX'] == 'No') & 
                       (missing_char['XRD'] == 'No') & 
                       (missing_char['XPS'] == 'No')).sum()
        
        print(f"\nOf the {missing_count} samples missing {CHARACTERIZATION_TYPE}:")
        if CHARACTERIZATION_TYPE != 'EDX':
            print(f"  • {has_edx} have EDX")
        if CHARACTERIZATION_TYPE != 'XRD':
            print(f"  • {has_xrd} have XRD")
        if CHARACTERIZATION_TYPE != 'XPS':
            print(f"  • {has_xps} have XPS")
        print(f"  • {has_nothing} have NO characterization at all")
        
        print("="*80)
        
        # Save to CSV
        output_file = f"missing_{CHARACTERIZATION_TYPE.lower()}.csv"
        missing_char.to_csv(output_file, index=False)
        print(f"\n💾 Results saved to: {output_file}")
        print(f"📁 Full path: {os.path.abspath(output_file)}")
        
    else:
        print(f"\n✅ All samples have {CHARACTERIZATION_TYPE}!")
    
else:
    print("❌ characterization_df not found. Please run the sputtering characterization table cell first.")

🔍 Searching for samples by Lena Mittmann
📊 Missing characterization type: XRD

📋 Found 179 samples WITHOUT XRD:

• mittma 0010 Cu - Position: BL
  └─ Has: EDX
• mittma 0010 Cu - Position: FR
  └─ Has: EDX
• mittma 0010 Cu - Position: FL
  └─ Has: EDX
• mittma 0010 Cu - Position: BR
  └─ Has: EDX
• mittma 0027 Cu - Position: FR
  └─ Has: EDX
• mittma 0027 Cu - Position: BR
  └─ Has: EDX
• mittma 0027 Cu - Position: FL
  └─ Has: EDX
• mittma 0027 Cu - Position: BL
  └─ Has: EDX
• mittma 0041 Cu - Position: BL
  └─ Has: EDX
• mittma 0041 Cu - Position: BR
  └─ Has: EDX
• mittma 0041 Cu - Position: FL
  └─ Has: EDX
• mittma 0041 Cu - Position: FR
  └─ Has: EDX
• mittma 0007 Cu - Position: FR
  └─ Has: EDX
• mittma 0007 Cu - Position: BL
  └─ Has: EDX
• mittma 0007 Cu - Position: BR
  └─ Has: EDX
• mittma 0007 Cu - Position: FL
  └─ Has: EDX
• mittma 0003 Cu - Position: FR
  └─ Has: EDX
• mittma 0003 Cu - Position: FL
  └─ Has: EDX
• mittma 0003 Cu - Position: BL
  └─ Has: EDX
• mittma 0006

### If you are not sure how to spell your name to NOMAD

In [28]:
# Query to get all authors from entries
all_authors = []
author_entries = {}  # Dictionary to store entries per author
page_after_value = None
total_entries = 0

print("Fetching authors from all entries...")

while True:
    query = {
        "owner": "visible",
        "pagination": {
            "page_size": 1000,
            "order_by": "entry_id",
            "page_after_value": page_after_value
        },
        "required": {
            "include": ["authors", "entry_id"]  # Get authors and entry_id
        }
    }

    try:
        response = requests.post(
            f"{URL}/entries/query",
            headers=headers,
            json=query,
            timeout=60,
        )
        response.raise_for_status()
        response_json = response.json()
        
        page_data = response_json.get('data', [])
        
        # Process each entry
        for entry in page_data:
            authors = entry.get('authors', [])
            entry_id = entry.get('entry_id')
            
            # Add to our tracking
            for author in authors:
                author_name = author.get('name', 'Unknown')
                if author_name not in author_entries:
                    author_entries[author_name] = []
                author_entries[author_name].append(entry_id)

        if not total_entries:
            total_entries = response_json.get('pagination', {}).get('total', 0)
            
        page_after_value = response_json.get('pagination', {}).get('next_page_after_value')
        
        print(f"  ... processed {sum(len(v) for v in author_entries.values())} entries / {total_entries} total")

        if not page_after_value or not page_data:
            break

    except Exception as e:
        print(f"❌ An error occurred during fetching: {e}")
        break

print("\nFetching complete.")
print("=" * 60)

# Display results
if author_entries:
    print(f"\n📊 Found {len(author_entries)} unique authors:\n")
    
    # Sort by number of entries (descending)
    sorted_authors = sorted(author_entries.items(), key=lambda x: len(x[1]), reverse=True)
    
    for author_name, entries in sorted_authors:
        print(f"• {author_name}: {len(entries)} entries")
    
    print("\n" + "=" * 60)
    print(f"Total unique authors: {len(author_entries)}")
    print(f"Total entries processed: {sum(len(v) for v in author_entries.values())}")
else:
    print("No author information found.")

Fetching authors from all entries...
  ... processed 1141 entries / 32267 total
  ... processed 2277 entries / 32267 total
  ... processed 3420 entries / 32267 total
  ... processed 4528 entries / 32267 total
  ... processed 5656 entries / 32267 total
  ... processed 6793 entries / 32267 total
  ... processed 7930 entries / 32267 total
  ... processed 9053 entries / 32267 total
  ... processed 10168 entries / 32267 total
  ... processed 11308 entries / 32267 total
  ... processed 12451 entries / 32267 total
  ... processed 13580 entries / 32267 total
  ... processed 14695 entries / 32267 total
  ... processed 15838 entries / 32267 total
  ... processed 16984 entries / 32267 total
  ... processed 18123 entries / 32267 total
  ... processed 19246 entries / 32267 total
  ... processed 20357 entries / 32267 total
  ... processed 21469 entries / 32267 total
  ... processed 22623 entries / 32267 total
  ... processed 23747 entries / 32267 total
  ... processed 24859 entries / 32267 total
  .